# ResNet50 Stage 3 — D3 External Inference Sustainability (Frozen Classification Checkpoints)

This is a **standalone ResNet50**, **not** the ResNet50 constituent used inside REMNET. The workflow reuses the uploaded `Resnet_co2(2).ipynb` Stage 2 sustainability reporting approach, the proven standalone ResNet50 classification checkpoint/manifest resolver, and the finalized Stage 3 D3 inference-monitoring protocol.

**No retraining or fine-tuning.** Five frozen standalone classification checkpoints are restored strictly (seeds 7, 49, 112, 345, 476). Each seed evaluates all **3,497** independent-screened D3 RealWaste images using `350 × 350` RGB, batch size **32**. Monitoring excludes checkpoint loading and warm-up. At least two complete passes and at least 35 seconds of monitored inference per seed are required. More complete passes are permitted if the minimum monitoring interval is not reached.

**Reporting:** original CodeCarbon emissions and standardized CO₂e (`energy kWh × 0.475 kg CO₂e/kWh`) stay in separate columns; means and sample SD (`ddof=1`) use fixed decimals. Historical Stage 2 ResNet50 training is reproduced *only as historical context* from the uploaded notebook's actual saved output, not rerun or added to Stage 3 D3 inference. All outputs and unaltered original CodeCarbon CSVs are exported in a single ZIP.

**Comparability note:** historical Stage 2 training CO₂ uses the CodeCarbon-reported estimate from that run, while standardized historical CO₂ applies the same fixed factor independently. Stage 2 training and Stage 3 D3 inference are different workloads and must not be presented as a matched energy comparison.


## 1. Fixed configuration and input files

In [2]:
from __future__ import annotations
import os, sys, json, hashlib, zipfile, subprocess, importlib.util, traceback
from pathlib import Path, PurePosixPath
from collections import Counter
from datetime import datetime, timezone

# Kaggle input location confirmed by the user; adjust ONLY if Kaggle moves the dataset.
INPUT_ROOT = Path('/kaggle/input/datasets/akash264208/manifist-part2/Section3')
OUTPUT = Path('/kaggle/working/ResNet50_D3_Inference_Sustainability')
OUTPUT.mkdir(parents=True, exist_ok=True)
SEEDS = (7, 49, 112, 345, 476)
MODEL_NAME = 'ResNet50'
CARBON_FACTOR = 0.475  # standardized kg CO2e / kWh (keep CodeCarbon original separately)
MIN_COMPLETE_PASSES = 2
MIN_MONITOR_SECONDS = 35.0
CODECARBON_SAMPLING_SECONDS = 15
ZIP_PATH = Path('/kaggle/working/ResNet50_D3_Inference_Sustainability_All_Outputs.zip')
CHECKPOINT_MODEL_METADATA = 'ResNet-50'
CHECKPOINT_ARCHITECTURE = 'torchvision.resnet50'
CLASSES = ('cardboard','glass','metal','paper','plastic','trash')
CLASS_TO_ID = {name:i for i,name in enumerate(CLASSES)}
EXPECTED_EXTERNAL_COUNT = 3497
EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT = '0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0'
# Set this to the exact Kaggle folder containing the D3 Section 2 CSV files.
# Example: '/kaggle/input/datasets/YOUR_ACCOUNT/YOUR_D3_SECTION2_OUTPUTS'
# If None, the notebook locates unique matching files under /kaggle/input.
D3_MANIFEST_ROOT = '/kaggle/input/datasets/akash264208/manifist-part2/Section3/manifist'
BATCH_SIZE = 32

# Optional genuine historical STANDALONE ResNet50 prediction files.
# Each file, when present, must have source_path and predicted_class_id columns,
# and preferably all six prob_CLASS columns. They must come from Stage 2.
# Set None to discover files automatically in the attached private Kaggle input.
HISTORICAL_REFERENCE_DIR = None
REQUIRE_HISTORICAL_REFERENCE = False  # Set True to STOP external evaluation if references are absent.

assert INPUT_ROOT.is_dir(), f'Input dataset was not attached: {INPUT_ROOT}'
print('Kaggle input:', INPUT_ROOT)
print('Output directory:', OUTPUT)
print('Seeds:', SEEDS)
print('Frozen standalone ResNet50 classification checkpoints, Stage 3 D3 sustainability; no training or fine-tuning.')

Kaggle input: /kaggle/input/datasets/akash264208/manifist-part2/Section3
Output directory: /kaggle/working/ResNet50_D3_Inference_Sustainability
Seeds: (7, 49, 112, 345, 476)
Frozen standalone ResNet50 classification checkpoints, Stage 3 D3 sustainability; no training or fine-tuning.


## 2. Dependencies; strict standalone ResNet50 classification-checkpoint inventory

In [3]:
# Install kagglehub only if unavailable; do not install or download pretrained weights.
for name in ('kagglehub', 'timm'):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', name])
import kagglehub, timm, re, numpy as np, pandas as pd, torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU inference selected')
print('torch:',torch.__version__)
print('torchvision:',__import__('torchvision').__version__)
print('kagglehub:',getattr(kagglehub,'__version__','installed'))

def only_directory_with_name(root, name):
    candidates = sorted(p for p in root.rglob('*') if p.is_dir() and p.name.casefold() == name.casefold())
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one directory named {name!r} under {root}; found {candidates}')
    return candidates[0]

def unique_by_filename(root, filename):
    matches = sorted(p for p in root.rglob(filename) if p.is_file())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename!r} in {root}, found {len(matches)}: {matches}')
    return matches[0]

CKPT_ROOT = only_directory_with_name(INPUT_ROOT, 'Classification_checkpoints')
STANDALONE_CKPT_DIR = only_directory_with_name(CKPT_ROOT, 'Resnet_50')
REMNET_CKPT_DIR = only_directory_with_name(CKPT_ROOT, 'Remnet')
RESULTS_ROOT = only_directory_with_name(INPUT_ROOT, 'per_seed_performance_summary')

def d3_csv_path(filename):
    root = Path(D3_MANIFEST_ROOT) if D3_MANIFEST_ROOT else Path('/kaggle/input')
    if not root.is_dir():
        raise FileNotFoundError(f'D3 manifest folder not attached: {root}')
    return unique_by_filename(root, filename)

EXTERNAL_MANIFEST = d3_csv_path('D3_Section2_Final_Automated_Screened_Six_Class_Manifest.csv')
CLASS_MAPPING = d3_csv_path('D3_Section2_Class_Mapping.csv')

# Only match the original standalone naming pattern from Resnet_50_Acc_s2.ipynb.
CHECKPOINTS = {}
for seed in SEEDS:
    expected_name = f'ResNet50_Torchvision_seed{seed}.pt'
    matches = sorted(p for p in STANDALONE_CKPT_DIR.rglob('*.pt')
                     if re.fullmatch(rf'ResNet50_Torchvision_seed{seed}(?:\(\d+\))?\.pt', p.name, re.IGNORECASE))
    if len(matches) != 1:
        raise RuntimeError(f'Expected precisely one standalone ResNet50 checkpoint for seed {seed} '
                           f'(preferred filename {expected_name}); found: {matches}')
    CHECKPOINTS[seed] = matches[0]

all_pts = sorted(CKPT_ROOT.rglob('*.pt'))
standalone_pts = sorted(STANDALONE_CKPT_DIR.rglob('*.pt'))
remnet_pts = sorted(REMNET_CKPT_DIR.rglob('*.pt'))
assert len(standalone_pts) == 5, f'Expected 5 standalone ResNet50 checkpoints, found {len(standalone_pts)}'
assert len(remnet_pts) == 15, f'Expected 15 REMNET ensemble constituent checkpoints, found {len(remnet_pts)}'
assert len(all_pts) == 30, f'Expected 30 high-accuracy checkpoints across all four configurations, found {len(all_pts)}'
assert not set(CHECKPOINTS.values()).intersection(remnet_pts)

# Optional original per-seed classification file; some supplied folders contain only
# ResNet50_5_Seed_Performance_Summary.csv (mean±sample SD, not individual-seed results).
HISTORICAL_PER_SEED = sorted(RESULTS_ROOT.rglob('ResNet50_Performance_Per_Seed.csv'))
HISTORICAL_SUMMARY = sorted(RESULTS_ROOT.rglob('ResNet50_5_Seed_Performance_Summary.csv'))
if len(HISTORICAL_PER_SEED)>1 or len(HISTORICAL_SUMMARY)>1:
    raise RuntimeError('Ambiguous historical Stage 2 ResNet50 result files')
HISTORICAL_PER_SEED = HISTORICAL_PER_SEED[0] if HISTORICAL_PER_SEED else None
HISTORICAL_SUMMARY = HISTORICAL_SUMMARY[0] if HISTORICAL_SUMMARY else None

print('Total .pt checkpoints:',len(all_pts),'| Standalone ResNet50:',len(standalone_pts),
      '| REMNET constituent checkpoints:',len(remnet_pts))
for seed,path in CHECKPOINTS.items(): print('STANDALONE ResNet50',seed,':',path)
for path in [EXTERNAL_MANIFEST, CLASS_MAPPING]: print('CSV:',path)
print('Historical per-seed file:',HISTORICAL_PER_SEED or 'not present')
print('Historical 5-seed summary:',HISTORICAL_SUMMARY or 'not present')

# Match the finalized REMNET Stage 3 tracking configuration; install only if needed.
from importlib.metadata import version as installed_version
try:
    _cc_version=installed_version('codecarbon')
except Exception:
    _cc_version=None
if _cc_version != '3.3.2':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','codecarbon==3.3.2'])
from codecarbon import EmissionsTracker
import time, shutil, platform
print('CodeCarbon:',installed_version('codecarbon'))


GPU: Tesla T4
torch: 2.11.0+cu130
torchvision: 0.26.0+cu130
kagglehub: 1.0.3
Total .pt checkpoints: 30 | Standalone ResNet50: 5 | REMNET constituent checkpoints: 15
STANDALONE ResNet50 7 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Resnet_50/ResNet50_Torchvision_seed7.pt
STANDALONE ResNet50 49 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Resnet_50/ResNet50_Torchvision_seed49.pt
STANDALONE ResNet50 112 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Resnet_50/ResNet50_Torchvision_seed112.pt
STANDALONE ResNet50 345 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Resnet_50/ResNet50_Torchvision_seed345.pt
STANDALONE ResNet50 476 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Resnet_50/ResNet50_Torchvision_seed476.pt
CSV: /kaggle/input/datasets/akash264208/manifist-part2/Section3/manifis

## 3. Validate the locked D3 manifest and six-class RealWaste mapping

In [4]:
def file_sha256(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path,'rb') as stream:
        for b in iter(lambda:stream.read(chunk_size),b''):
            h.update(b)
    return h.hexdigest()

def canonical_external_fingerprint(df):
    h = hashlib.sha256()
    for row in df.sort_values('external_relative_path').itertuples(index=False):
        h.update((row.external_relative_path+'|'+str(int(row.output_index))+'|'+
                  row.sha256+'\n').encode('utf-8'))
    return h.hexdigest()

external_df = pd.read_csv(EXTERNAL_MANIFEST, dtype=str, keep_default_na=False)
mapping_df = pd.read_csv(CLASS_MAPPING, dtype=str, keep_default_na=False)

assert set(['external_relative_path','class','output_index','sha256','final_status']) <= set(external_df.columns)
assert set(['d3_source_label','remnet_trained_class','remnet_output_index','include_for_primary_d3']) <= set(mapping_df.columns)
assert len(external_df)==EXPECTED_EXTERNAL_COUNT, 'Unexpected D3 manifest count (should be 3497).'
assert external_df['sha256'].nunique()==EXPECTED_EXTERNAL_COUNT, 'Duplicate hashes in external manifest.'
assert external_df['external_relative_path'].nunique()==EXPECTED_EXTERNAL_COUNT
assert set(external_df['class']) == set(CLASSES)
assert external_df['final_status'].eq('ELIGIBLE_AFTER_REFERENCE_VERIFICATION').all(), 'Noneligible images in frozen manifest.'
included_mapping = mapping_df.loc[mapping_df['include_for_primary_d3'].str.lower().eq('true')].copy()
assert len(included_mapping) == 6
assert set(included_mapping['remnet_trained_class']) == set(CLASSES)
for item in included_mapping.itertuples(index=False):
    assert int(item.remnet_output_index) == CLASS_TO_ID[item.remnet_trained_class]
    if 'class_review' in mapping_df.columns:
        assert item.class_review == 'CONFIRMED_BY_USER'
assert set(external_df['source_category']) == set(included_mapping['d3_source_label'])
assert all(int(i)==CLASS_TO_ID[c] for c,i in zip(external_df['class'],external_df['output_index']))

EXTERNAL_FINGERPRINT=canonical_external_fingerprint(external_df)
assert EXTERNAL_FINGERPRINT == EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT, (EXTERNAL_FINGERPRINT,'External cohort fingerprint mismatch')
print('D3 frozen manifest verified:',len(external_df),'Fingerprint:',EXTERNAL_FINGERPRINT)
print('Six class mapping:',CLASS_TO_ID)
print('External per-class counts:',external_df['class'].value_counts().sort_index().to_dict())

D3 frozen manifest verified: 3497 Fingerprint: 0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0
Six class mapping: {'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}
External per-class counts: {'cardboard': 454, 'glass': 419, 'metal': 757, 'paper': 500, 'plastic': 889, 'trash': 478}


## 4. Load the attached RealWaste D3 dataset

In [5]:
# Stage 3 D3 dataset path — fixed mounted Kaggle input.
EXTERNAL_HANDLE = 'akash264208/real-waste'
external_download = Path('/kaggle/input/datasets/akash264208/real-waste')
external_download_method = 'attached Kaggle dataset'
assert external_download.is_dir(), f'D3 RealWaste dataset missing: {external_download}'
print('D3 RealWaste:', external_download)

D3 RealWaste: /kaggle/input/datasets/akash264208/real-waste


## 5. Resolve D3 images and verify every locked image hash

In [6]:
IMG_EXT={'.jpg','.jpeg','.png','.webp','.bmp'}

def safe_parts(relative):
    p=PurePosixPath(str(relative).replace('\\','/'))
    if p.is_absolute() or not p.parts or any(s in ('','.','..') for s in p.parts):
        raise ValueError(f'Unsafe image relative path: {relative!r}')
    return p.parts

def external_class_roots(root):
    expected=set(included_mapping['d3_source_label'])
    candidates=[root] + sorted((p for p in root.rglob('*') if p.is_dir()), key=str)
    return [p for p in candidates if all((p/c).is_dir() for c in expected)]

def resolve_external(external,download):
    candidates=external_class_roots(download)
    print('D3 RealWaste class-root candidates:',candidates)
    if not candidates: raise FileNotFoundError('Missing RealWaste source-category directories.')
    expected_rows=[(safe_parts(r.external_relative_path),r.sha256,r.source_category)
                   for _,r in external.iterrows()]
    for root in candidates:
        try:
            dirs=[p for p in root.iterdir() if p.is_dir()]
            image_counts={p.name:sum(x.is_file() and x.suffix.lower() in IMG_EXT
                         for x in p.rglob('*')) for p in dirs}
            if len(image_counts)!=9 or sum(image_counts.values())!=4752:
                raise RuntimeError(f'Expected nine RealWaste folders / 4752 images: {image_counts}')
            if sum(image_counts[c] for c in included_mapping['d3_source_label'])!=3587:
                raise RuntimeError('D3 six-category source total differs from Section 2.')
            resolved=[]
            for parts,digest,source_cls in expected_rows:
                image=root.joinpath(*parts)
                if not image.is_file() or parts[0]!=source_cls:
                    raise FileNotFoundError(str(image))
                if file_sha256(image)!=digest:
                    raise ValueError(f'D3 image hash mismatch: {image}')
                resolved.append(str(image))
            print('D3 RealWaste images SHA-256 verified:',len(resolved),'root:',root)
            return resolved,root,image_counts
        except (FileNotFoundError,ValueError,RuntimeError) as exc:
            print('D3 candidate rejected:',root,'reason:',str(exc)[:180])
    raise RuntimeError('No RealWaste root matches the 3497 locked image hashes.')

external_paths, external_image_root, d3_counts=resolve_external(external_df,external_download)
external_df=external_df.copy(); external_df['resolved_path']=external_paths
external_eval_df=external_df.sort_values('external_relative_path').reset_index(drop=True)
assert len(external_eval_df)==3497
print('External evaluated images:',len(external_eval_df))

D3 RealWaste class-root candidates: [PosixPath('/kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste')]
D3 RealWaste images SHA-256 verified: 3497 root: /kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste
External evaluated images: 3497


## 6. Restore the standalone `ResNet50_Model` wrapper, frozen Stage 2 weights, and preprocessing

In [7]:
TEST_TRANSFORM = transforms.Compose([
    transforms.Resize((350,350)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3,(0.5,)*3),
])

class ResNet50_Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = models.resnet50(weights=None)
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

def construct_resnet50():
    return ResNet50_Model(num_classes=len(CLASSES))

class LockedImageDataset(Dataset):
    def __init__(self,df,kind):
        self.df=df.reset_index(drop=True)
        self.kind=kind
        self.path_col='external_relative_path'
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i]
        with Image.open(row['resolved_path']) as im:
            x=TEST_TRANSFORM(im.convert('RGB'))
        return x,int(CLASS_TO_ID[row['class']]),i

def locked_loader(df,kind):
    return DataLoader(LockedImageDataset(df,kind),batch_size=BATCH_SIZE,
                      shuffle=False,num_workers=0,pin_memory=(DEVICE.type=='cuda'))

def restore_checkpoint(seed):
    p=CHECKPOINTS[seed]
    # PyTorch weights_only=True is appropriate for the Stage 2 dictionary format.
    ckpt=torch.load(p,map_location='cpu',weights_only=True)
    assert isinstance(ckpt,dict) and 'state_dict' in ckpt, f'Unexpected checkpoint format: {p}'
    assert str(ckpt.get('model'))==CHECKPOINT_MODEL_METADATA, f'Model mismatch in {p.name}: {ckpt.get("model")}'
    assert int(ckpt.get('seed'))==seed, f'Seed mismatch: {p.name}'
    assert ckpt.get('architecture')==CHECKPOINT_ARCHITECTURE, f'Architecture mismatch: {p.name}'
    state=ckpt['state_dict']
    assert tuple(state['model.fc.weight'].shape)==(6,2048), f'Unexpected ResNet head shape in {p.name}'
    model=construct_resnet50()
    model.load_state_dict(state,strict=True)
    model.to(DEVICE).eval()
    for param in model.parameters():param.requires_grad_(False)
    return model,ckpt,file_sha256(p)

def infer(model,df,kind):
    labels=[]; probs=[]; positions=[]
    with torch.inference_mode():
        for images,y,index in locked_loader(df,kind):
            pred=torch.softmax(model(images.to(DEVICE,non_blocking=True)),dim=1)
            probs.append(pred.cpu().numpy().astype(np.float64))
            labels.append(y.numpy()); positions.append(index.numpy())
    labels=np.concatenate(labels).astype(np.int64)
    probs=np.concatenate(probs)
    positions=np.concatenate(positions)
    assert np.array_equal(positions,np.arange(len(df))), 'Image order changed during inference.'
    assert np.array_equal(labels,df['class'].map(CLASS_TO_ID).to_numpy()), 'Label order mismatch.'
    assert probs.shape==(len(df),6) and np.isfinite(probs).all(), 'Invalid six-class probability array.'
    assert np.allclose(probs.sum(axis=1),1.0,atol=2e-6), 'Probabilities do not sum to one.'
    return labels,probs

# Metadata/strict-load smoke test before expensive multi-seed inference.
_smoke, _smoke_info, _smoke_digest=restore_checkpoint(SEEDS[0])
with torch.inference_mode():
    _test_logits=_smoke(torch.zeros(1,3,350,350,device=DEVICE))
assert tuple(_test_logits.shape)==(1,6)
print('ResNet50 strict-load and six-logit smoke test: PASS')
del _smoke,_smoke_info,_test_logits
if DEVICE.type=='cuda':torch.cuda.empty_cache()

ResNet50 strict-load and six-logit smoke test: PASS


## 7. CodeCarbon measurement boundaries and consistent timing

Use the Stage 2 **standalone ResNet50 classification checkpoints** (`ResNet50_Torchvision_seed*.pt`), one per seed, and strictly verify state-dictionary keys, six-class output, model type, architecture, seed metadata and file hashes. No torchvision pretrained download occurs.

Start tracking **after** loading and two warm-up forward passes. Synchronize CUDA before and after the monitored workload. Every complete pass processes all 3,497 D3 samples exactly once, batch size 32. Run at least two passes and continue by *whole passes* if 35 seconds has not elapsed. Tracking includes data loading/preprocessing but excludes checkpoint load and warm-up, consistent with REMNET monitoring. Compute per-image costs from **all processed images**, not unique images. Original CodeCarbon emissions and standardized factor-based CO₂e must remain separate.


In [8]:
LOG_ROOT = OUTPUT/'CodeCarbon_Logs'
LOG_ROOT.mkdir(parents=True,exist_ok=True)

def make_tracker(log_dir, seed):
    log_dir.mkdir(parents=True,exist_ok=True)
    prior = log_dir/'emissions.csv'
    if prior.exists():
        prior.rename(log_dir/f'emissions_previous_{time.time_ns()}.csv')
    options = dict(
        project_name=f'ResNet50_Stage3_seed{seed}_External3497',
        output_dir=str(log_dir), output_file='emissions.csv',
        save_to_file=True, tracking_mode='machine',
        measure_power_secs=CODECARBON_SAMPLING_SECONDS,
        pue=1.0, log_level='error'
    )
    if DEVICE.type=='cuda': options['gpu_ids']=[0]
    return EmissionsTracker(**options)

def gpu_sync():
    if DEVICE.type=='cuda': torch.cuda.synchronize()

def measured_row(log_dir):
    filepath=log_dir/'emissions.csv'
    if not filepath.is_file():
        raise FileNotFoundError(f'CodeCarbon emissions file missing: {filepath}')
    df=pd.read_csv(filepath)
    if df.empty: raise RuntimeError(f'Empty CodeCarbon file: {filepath}')
    row=df.iloc[-1].to_dict()
    energy=float(row['energy_consumed'])
    if not np.isfinite(energy) or energy<0: raise RuntimeError(f'Invalid CodeCarbon energy {energy}')
    emissions=float(row['emissions']) if 'emissions' in row and pd.notna(row['emissions']) else np.nan
    return row,energy,emissions

def monitored_resnet50_inference(seed, model):
    loader=locked_loader(external_eval_df,'external')
    model.eval()
    # Two identical warm-up forward-only batches, before monitoring starts.
    images, _, _ = next(iter(loader))
    images=images.to(DEVICE,non_blocking=True)
    with torch.inference_mode():
        for _ in range(2):
            p=torch.softmax(model(images),dim=1)
            if p.shape[1]!=6: raise RuntimeError('Unexpected six-class output during warm-up')
    gpu_sync()
    del images,p
    log_dir=LOG_ROOT/f'ResNet50_seed{seed}'/'inference'
    tracker=make_tracker(log_dir,seed)
    passes=0
    processed=0
    labels_first=[]; pred_first=[]
    started=False
    try:
        tracker.start()
        started=True
        gpu_sync()
        t0=time.perf_counter()
        with torch.inference_mode():
            while passes<MIN_COMPLETE_PASSES or (time.perf_counter()-t0)<MIN_MONITOR_SECONDS:
                current_positions=[]
                for images,y,positions in loader:
                    images=images.to(DEVICE,non_blocking=True)
                    probs=torch.softmax(model(images),dim=1)
                    if tuple(probs.shape)!=(len(y),6):
                        raise RuntimeError('Six-class output mismatch')
                    if passes==0:
                        labels_first.extend(y.tolist())
                        pred_first.extend(probs.argmax(dim=1).cpu().tolist())
                        current_positions.extend(positions.tolist())
                    processed+=len(y)
                if passes==0 and current_positions!=list(range(len(external_eval_df))):
                    raise RuntimeError('External image order changed during monitored inference')
                passes+=1
        gpu_sync()
        seconds=time.perf_counter()-t0
    finally:
        if started:
            gpu_sync()
            tracker.stop()
    if processed!=passes*len(external_eval_df):
        raise RuntimeError(f'Incomplete inference pass: {processed} images / {passes} passes')
    row,energy,cc_emissions=measured_row(log_dir)
    if seconds<=0 or processed<=0:raise RuntimeError('Invalid measured workload')
    ytrue=np.asarray(labels_first,dtype=int)
    ypred=np.asarray(pred_first,dtype=int)
    expected=external_eval_df['class'].map(CLASS_TO_ID).to_numpy()
    assert np.array_equal(ytrue,expected)
    accuracy_pct=100*float((ytrue==ypred).mean())
    return {
       'model':'ResNet50','seed':int(seed),
       'unique_external_images':len(external_eval_df),
       'number_complete_repetitions':passes,'total_processed_images':processed,
       'batch_size':BATCH_SIZE,'input_resolution':'350x350',
       'monitoring_seconds':seconds,'total_inference_energy_kWh':energy,
       'codecarbon_emissions_kgCO2e':cc_emissions,
       'standardized_CO2e_kg_fixed_factor':energy*CARBON_FACTOR,
       'standardized_CO2_factor_kg_per_kWh':CARBON_FACTOR,
       'energy_kWh_per_processed_image':energy/processed,
       'time_seconds_per_processed_image':seconds/processed,
       'throughput_images_per_second':processed/seconds,
       'first_pass_accuracy_pct_diagnostic':accuracy_pct,
       'codecarbon_duration_seconds':row.get('duration',np.nan),
       'codecarbon_energy_column':'energy_consumed',
       'codecarbon_version':installed_version('codecarbon'),
       'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU',
       'raw_emissions_csv':str(log_dir/'emissions.csv')
    }


## 8. Five-seed standalone ResNet50 D3 external inference; checkpoint registry and original CodeCarbon logs

In [9]:
sustainability_rows=[]
checkpoint_rows=[]
for seed in SEEDS:
    print(f'\n=== STANDALONE RESNET50 STAGE 3 D3 INFERENCE — SEED {seed} ===',flush=True)
    model,meta,digest=restore_checkpoint(seed)
    checkpoint_rows.append({
        'model':'ResNet50','seed':seed,'checkpoint_path':str(CHECKPOINTS[seed]),
        'checkpoint_file_sha256':digest,'checkpoint_seed':meta.get('seed'),
        'checkpoint_model':meta.get('model'), 'checkpoint_architecture':meta.get('architecture'),
        'split_sha256':meta.get('split_sha256'),
        'classification_checkpoint_source':'Resnet_50/ResNet50_Torchvision_seed*.pt',
        'preprocessing':'RGB Resize(350,350) ToTensor Normalize(.5,.5,.5)'})
    result=monitored_resnet50_inference(seed,model)
    sustainability_rows.append(result)
    pd.DataFrame(sustainability_rows).to_csv(OUTPUT/'ResNet50_Stage3_D3_Inference_Per_Seed.csv',index=False)
    pd.DataFrame(checkpoint_rows).to_csv(OUTPUT/'ResNet50_Stage3_D3_Classification_Checkpoint_Registry.csv',index=False)
    print(f"Seed {seed} completed | D3 external: {result['unique_external_images']} unique images × "
          f"{result['number_complete_repetitions']} complete passes = {result['total_processed_images']} processed "
          f"| energy {result['total_inference_energy_kWh']:.9f} kWh "
          f"| original CodeCarbon CO2e {result['codecarbon_emissions_kgCO2e']:.9f} kg "
          f"| standardized CO2e {result['standardized_CO2e_kg_fixed_factor']:.9f} kg "
          f"| monitored runtime {result['monitoring_seconds']:.3f} s",flush=True)
    del model
    if DEVICE.type=='cuda':torch.cuda.empty_cache()

checkpoint_df=pd.DataFrame(checkpoint_rows)
checkpoint_df.to_csv(OUTPUT/'ResNet50_Stage3_D3_Classification_Checkpoint_Registry.csv',index=False)



=== STANDALONE RESNET50 STAGE 3 D3 INFERENCE — SEED 7 ===


[codecarbon WARNING @ 12:02:54] Multiple instances of codecarbon are allowed to run at the same time.


Seed 7 completed | D3 external: 3497 unique images × 2 complete passes = 6994 processed | energy 0.002021969 kWh | original CodeCarbon CO2e 0.000334726 kg | standardized CO2e 0.000960436 kg | monitored runtime 92.055 s

=== STANDALONE RESNET50 STAGE 3 D3 INFERENCE — SEED 49 ===
Seed 49 completed | D3 external: 3497 unique images × 2 complete passes = 6994 processed | energy 0.002146171 kWh | original CodeCarbon CO2e 0.000355287 kg | standardized CO2e 0.001019431 kg | monitored runtime 95.010 s

=== STANDALONE RESNET50 STAGE 3 D3 INFERENCE — SEED 112 ===
Seed 112 completed | D3 external: 3497 unique images × 2 complete passes = 6994 processed | energy 0.002179065 kWh | original CodeCarbon CO2e 0.000360732 kg | standardized CO2e 0.001035056 kg | monitored runtime 96.291 s

=== STANDALONE RESNET50 STAGE 3 D3 INFERENCE — SEED 345 ===
Seed 345 completed | D3 external: 3497 unique images × 2 complete passes = 6994 processed | energy 0.002191941 kWh | original CodeCarbon CO2e 0.000362864 kg |

## 9. Inference results: mean ± sample SD across five seeds (`ddof=1`), fixed-decimal output

In [10]:
results_df=pd.DataFrame(sustainability_rows)
assert len(results_df)==5 and set(results_df.seed)==set(SEEDS)
MEASURES=[
    'total_inference_energy_kWh','codecarbon_emissions_kgCO2e',
    'standardized_CO2e_kg_fixed_factor','monitoring_seconds',
    'energy_kWh_per_processed_image','time_seconds_per_processed_image',
    'throughput_images_per_second','first_pass_accuracy_pct_diagnostic']
summary_rows=[]
for measure in MEASURES:
    values=results_df[measure].astype(float)
    summary_rows.append({'model':'ResNet50','measure':measure,'n_seeds':len(values),
                         'mean':values.mean(),'sample_sd':values.std(ddof=1),
                         'min':values.min(),'max':values.max()})
summary_df=pd.DataFrame(summary_rows)
results_df.to_csv(OUTPUT/'ResNet50_Stage3_D3_Inference_Per_Seed.csv',index=False,float_format='%.12f')
summary_df.to_csv(OUTPUT/'ResNet50_Stage3_D3_Inference_Five_Seed_Mean_SD.csv',index=False,float_format='%.12f')
print(summary_df[['measure','mean','sample_sd']].to_string(index=False,
    formatters={'mean':lambda v:f'{v:.12f}','sample_sd':lambda v:f'{v:.12f}'}))
print('Five-seed D3 inference summary saved.')


                           measure            mean      sample_sd
        total_inference_energy_kWh  0.002144345746 0.000070555024
       codecarbon_emissions_kgCO2e  0.000354984499 0.000011679991
 standardized_CO2e_kg_fixed_factor  0.001018564229 0.000033513636
                monitoring_seconds 95.362768193000 1.982480171567
    energy_kWh_per_processed_image  0.000000306598 0.000000010088
  time_seconds_per_processed_image  0.013634939690 0.000283454414
      throughput_images_per_second 73.366880607957 1.556769708330
first_pass_accuracy_pct_diagnostic 49.974263654561 4.452553970585
Five-seed D3 inference summary saved.


## 10. Historical Stage 2 standalone ResNet50 training — reference only

The following rows come from the **saved final output of the supplied `Resnet_co2(2).ipynb`**, not from new training. Original Stage 2 CodeCarbon CO₂ is reported verbatim at the precision printed in that notebook. A separate standardized CO₂e value is computed using **0.475 kg CO₂e/kWh**; never replace the original CodeCarbon emissions. Historical Stage 2 training monitored training and epoch-wise evaluation; Stage 3 monitored D3 external inference only. These are not matched-workload comparisons.

Stage 2 displayed five-seed means in the uploaded notebook were `0.02338760066016622` kWh, `0.0067274041086501` kg original CodeCarbon CO₂e and `15.218582243919371` min, based on higher-precision underlying records. The per-seed rows below are rounded values from the notebook's printed table; sample SDs computed from these are approximate.


In [11]:
# Historical Stage 2 values from actual saved outputs in uploaded Resnet_co2(2).ipynb.
# Stage 2 CO2 values are ORIGINAL CodeCarbon estimates, not 0.475-standardized.
historical_rows=[
    {'seed':7,  'stage2_codecarbon_CO2e_kg_reported':0.007051,'training_energy_kWh_reported':0.024511,'training_time_min_reported':16.000540},
    {'seed':49, 'stage2_codecarbon_CO2e_kg_reported':0.006300,'training_energy_kWh_reported':0.021900,'training_time_min_reported':14.246001},
    {'seed':112,'stage2_codecarbon_CO2e_kg_reported':0.007022,'training_energy_kWh_reported':0.024412,'training_time_min_reported':15.883103},
    {'seed':345,'stage2_codecarbon_CO2e_kg_reported':0.006978,'training_energy_kWh_reported':0.024258,'training_time_min_reported':15.753303},
    {'seed':476,'stage2_codecarbon_CO2e_kg_reported':0.006287,'training_energy_kWh_reported':0.021857,'training_time_min_reported':14.209964},
]
historical_training_df=pd.DataFrame(historical_rows)
historical_training_df['stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy']=(
    historical_training_df['training_energy_kWh_reported']*CARBON_FACTOR)
historical_training_df['stage2_original_source']='Resnet_co2(2).ipynb, saved output (rounded printed per-seed table)'
historical_training_df['status']='Historical Stage 2 standalone ResNet50; not remeasured in Stage 3'
historical_training_df.to_csv(OUTPUT/'ResNet50_Stage2_Historical_Training_Per_Seed.csv',
                              index=False,float_format='%.12f')
reported_historical_means={
    'stage2_codecarbon_CO2e_kg_reported':0.0067274041086501,
    'training_energy_kWh_reported':0.02338760066016622,
    'training_time_min_reported':15.218582243919371,
}
historical_summary=[]
for measure in ['stage2_codecarbon_CO2e_kg_reported','training_energy_kWh_reported',
                'training_time_min_reported','stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy']:
    vals=historical_training_df[measure].astype(float)
    historical_summary.append({
        'measure':measure,'n_seeds':5,
        'mean_as_reported_stage2_or_from_rounded_data':reported_historical_means.get(measure,float(vals.mean())),
        'mean_from_printed_rounded_seed_rows':float(vals.mean()),
        'sample_sd_from_printed_rounded_seed_rows':float(vals.std(ddof=1)),
        'note':'Standardized Stage 2 CO2 and SD estimated from printed rounded rows; other means retained from original notebook' })
historical_summary_df=pd.DataFrame(historical_summary)
historical_summary_df.to_csv(OUTPUT/'ResNet50_Stage2_Historical_Training_Mean_SD.csv',
                             index=False,float_format='%.12f')

print('\n'+'='*75)
print('HISTORICAL STAGE 2 — STANDALONE RESNET50 TRAINING (NOT MEASURED IN STAGE 3)')
print('='*75)
print(historical_training_df[['seed','stage2_codecarbon_CO2e_kg_reported',
       'training_energy_kWh_reported','stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy',
       'training_time_min_reported']].to_string(index=False,formatters={
       'stage2_codecarbon_CO2e_kg_reported':lambda v:f'{v:.9f}',
       'training_energy_kWh_reported':lambda v:f'{v:.9f}',
       'stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy':lambda v:f'{v:.9f}',
       'training_time_min_reported':lambda v:f'{v:.6f}',
    }))
print('Historical Stage 2 original CodeCarbon CO2e mean (kg):',f"{reported_historical_means['stage2_codecarbon_CO2e_kg_reported']:.12f}")
print('Historical Stage 2 training energy mean (kWh):',f"{reported_historical_means['training_energy_kWh_reported']:.12f}")
print('Historical Stage 2 training time mean (min):',f"{reported_historical_means['training_time_min_reported']:.9f}")
print('Historical Stage 2 standardized CO2e from rounded seed energies (kg):',
      f"{historical_training_df['stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy'].mean():.12f}")
print('Stage 2 historical data is separate from Stage 3 D3 measured inference; no combined-cost claim.')

print('\n'+'='*75)
print('STAGE 3 — D3 EXTERNAL RESNET50 INFERENCE (NEWLY MONITORED; 3,497 UNIQUE IMAGES)')
print('='*75)
print(results_df[['seed','unique_external_images','number_complete_repetitions','total_processed_images',
    'total_inference_energy_kWh','codecarbon_emissions_kgCO2e',
    'standardized_CO2e_kg_fixed_factor','monitoring_seconds',
    'energy_kWh_per_processed_image','time_seconds_per_processed_image',
    'throughput_images_per_second']].to_string(index=False,formatters={
       'total_inference_energy_kWh':lambda v:f'{v:.9f}',
       'codecarbon_emissions_kgCO2e':lambda v:f'{v:.9f}',
       'standardized_CO2e_kg_fixed_factor':lambda v:f'{v:.9f}',
       'monitoring_seconds':lambda v:f'{v:.3f}',
       'energy_kWh_per_processed_image':lambda v:f'{v:.12f}',
       'time_seconds_per_processed_image':lambda v:f'{v:.9f}',
       'throughput_images_per_second':lambda v:f'{v:.3f}',
    }))
print('Original CodeCarbon emissions and standardized emissions at 0.475 kg CO2e/kWh remain separate.')
print('Five-seed mean standardized CO2e (kg):',
      f"{results_df['standardized_CO2e_kg_fixed_factor'].mean():.9f}")
print('Five-seed sample SD standardized CO2e (kg):',
      f"{results_df['standardized_CO2e_kg_fixed_factor'].std(ddof=1):.9f}")



HISTORICAL STAGE 2 — STANDALONE RESNET50 TRAINING (NOT MEASURED IN STAGE 3)
 seed stage2_codecarbon_CO2e_kg_reported training_energy_kWh_reported stage2_standardized_CO2e_kg_fixed_0475_from_rounded_energy training_time_min_reported
    7                        0.007051000                  0.024511000                                                0.011642725                  16.000540
   49                        0.006300000                  0.021900000                                                0.010402500                  14.246001
  112                        0.007022000                  0.024412000                                                0.011595700                  15.883103
  345                        0.006978000                  0.024258000                                                0.011522550                  15.753303
  476                        0.006287000                  0.021857000                                                0.010382075               

## 11. Provenance configuration, full raw logs, and one downloadable ZIP

In [12]:
configuration={
 'experiment':'Standalone ResNet50 Stage 3 D3 external inference sustainability',
 'stage2_sustainability_source':'Resnet_co2(2).ipynb, historical saved notebook outputs',
 'historical_stage2_training':'Reference only; not remeasured or combined with D3 external inference',
 'historical_stage2_original_CO2':'Original CodeCarbon results reported in Stage 2 notebook',
 'historical_stage2_standardized_CO2':'Rounded historical energy times 0.475; distinct from original emissions',
 'checkpoint_source':'Frozen Stage 2 standalone ResNet50 classification checkpoints',
 'input_checkpoint_root':str(STANDALONE_CKPT_DIR),
 'not_ensemble_constituent_checkpoints':True,
 'seeds':list(SEEDS),'class_to_index':CLASS_TO_ID,
 'external_dataset':EXTERNAL_HANDLE,
 'd3_manifest_csv_path':str(EXTERNAL_MANIFEST),
 'external_unique_images':len(external_eval_df),
 'external_manifest_fingerprint':EXTERNAL_FINGERPRINT,
 'd1_original_test_evaluation_performed':False,
 'preprocessing':'RGB Resize((350,350)) ToTensor Normalize((0.5,)*3,(0.5,)*3)',
 'architecture':'ResNet50_Model wrapper / torchvision.models.resnet50(weights=None) / fc 2048->6',
 'no_training_no_adaptation':True,'classification_checkpoints_unmodified':True,
 'checkpoint_loading':'strict state_dict / model / architecture / seed metadata checks',
 'batch_size':BATCH_SIZE,'minimum_complete_passes':MIN_COMPLETE_PASSES,
 'minimum_monitor_seconds':MIN_MONITOR_SECONDS,
 'codecarbon_sampling_seconds':CODECARBON_SAMPLING_SECONDS,
 'codecarbon_version':installed_version('codecarbon'),
 'device':str(DEVICE),
 'hardware_gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None,
 'carbon_factor_fixed_kgCO2e_per_kWh':CARBON_FACTOR,
 'measurement_boundary':'After checkpoint loading and warm-up; full D3 passes, including batch loading/preprocessing; GPU synchronized for timing; no checkpoint load/warm-up in monitored interval.',
 'original_codecarbon_emissions':'Original emissions.csv emissions column, never replaced',
 'standardized_CO2e':'Energy in kWh times fixed carbon-intensity factor; separately displayed',
 'external_3497_test_sha256_verified':True,
 'fair_comparison_note':'Use identical hardware, batch 32, preprocessing and monitoring protocol; confirm classification checkpoint SHA256 registry matches frozen ResNet50 D3 classification run.',
 'created_utc':datetime.now(timezone.utc).isoformat()
}
(OUTPUT/'ResNet50_Stage3_D3_Inference_Configuration.json').write_text(
    json.dumps(configuration,indent=2,ensure_ascii=False),encoding='utf-8')
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(OUTPUT.rglob('*')):
        if p.is_file() and 'original_source_extracted' not in p.parts:
            z.write(p,p.relative_to(OUTPUT))
print('One downloadable archive:',ZIP_PATH)
print('Archive bytes:',ZIP_PATH.stat().st_size)
print('Outputs: standalone ResNet50 Stage 3 D3 inference five-seed CSVs, original CodeCarbon logs, checkpoint registry, historical standalone ResNet50 Stage 2 tables, configuration JSON.')


One downloadable archive: /kaggle/working/ResNet50_D3_Inference_Sustainability_All_Outputs.zip
Archive bytes: 8804
Outputs: standalone ResNet50 Stage 3 D3 inference five-seed CSVs, original CodeCarbon logs, checkpoint registry, historical standalone ResNet50 Stage 2 tables, configuration JSON.
